# Modelo bayesiano jerárquico de serie interrumpida

Implementa la sugerencia del profesor C. Pieringer: una **regresión lineal bayesiana jerárquica**, análoga al tutorial de NumPyro sobre fibrosis pulmonar, donde cada **establecimiento** cumple el papel de un paciente y cada **año** el de una medición.

**Requiere** `pip install numpyro jax` en el entorno `tesis_simce`. Corre en CPU (unos 3-4 minutos); no necesita GPU.

### Adaptación del modelo

La recta simple del tutorial no basta: la trayectoria escolar tiene un **quiebre** en 2022. Para cada establecimiento $i$ y año $t$:

$$y_{it} = a_i + d_i\,\text{POST}_t + b_i\,(t-2022)\,\text{POST}_t + p_{g(i)}\,(t-2016{,}5)\,\text{PRE}_t + \varepsilon_{it}$$

| Parámetro | Significado | Nivel |
|---|---|---|
| $a_i$ | nivel pre-pandemia | escuela |
| $d_i$ | **caída inicial** (salto en 2022) | escuela, con media por GSE |
| $b_i$ | **recuperación** (pendiente desde 2022) | escuela, con media por GSE |
| $p_g$ | **tendencia previa** a la pandemia | grupo socioeconómico |

El término $p_g$ contrasta un supuesto que el modelo simple da por hecho: que el nivel previo era plano. Si un grupo venía en alza, su "recuperación" aparente incluiría esa tendencia.

> **Revisión del 24-oct-2026 (auditoría de cuadernos y código).** El modelo no cambia y sus resultados tampoco. Se agregan dos verificaciones:

1. **Convergencia por escuela.** El criterio C2 se revisaba solo en los parámetros de grupo y de varianza. La nueva celda de la sección 2 revisa también la caída y la recuperación de cada una de las 2.861 escuelas. Salida esperada: R-hat máximo cerca de 1,003 y tamaño efectivo mínimo cerca de 1.900.
2. **Sensibilidad al GSE previo a la pandemia** (sección 10, nueva). El cuaderno asigna a cada escuela el GSE del año más reciente, que es posterior a 2022. 719 de las 2.861 escuelas (25 %) tienen un GSE modal 2015–2018 distinto. La sección 10 reajusta el modelo con el GSE previo. Salida esperada: GSE Bajo con caída +0,7 [−0,8; 2,1] frente a su promedio previo, tendencia previa 1,9 [1,5; 2,3] y caída frente a su tendencia de −9,8 puntos; GSE Alto −10,4. El hallazgo principal se mantiene.

La sección 10 agrega unos 4 minutos de cómputo.

In [1]:
import sys
from pathlib import Path
for m in list(sys.modules):
    if m == "simce_panel" or m.startswith("simce_panel."):
        del sys.modules[m]
candidatos = [Path.cwd() / "simce_panel" / "src", Path.cwd().parent / "src", Path.cwd() / "src"]
RUTA_SRC = next((p for p in candidatos if (p / "simce_panel" / "__init__.py").exists()), None)
sys.path.insert(0, str(RUTA_SRC))

import re
import numpy as np
import pandas as pd
import simce_panel as sp
B = sp.bayes
sp.configurar_logging()
cfg = sp.Config.desde_base(".")

## 1. Panel largo escuela-año

In [2]:
ancho = sp.io_datos.leer_csv(cfg.ruta_salida("simce_rbd_ancho_reducido_2015_2025.csv"))
ANIOS = [2015, 2016, 2017, 2018, 2022, 2023, 2024, 2025]
largo = pd.concat([pd.DataFrame({
    "rbd": ancho["rbd"], "cod_grupo_cat": ancho["cod_grupo_cat"], "anio": t,
    "y": pd.to_numeric(ancho[f"prom_mate2m_rbd_{t}"], errors="coerce"),
    "n": pd.to_numeric(ancho[f"nalu_mate2m_rbd_{t}"], errors="coerce")})
    for t in ANIOS]).dropna(subset=["y"])

# Modelo de medición: col_n activa la varianza sigma_w^2 / n + sigma_e^2.
# exigir_2022 hace explícito el criterio del comentario 22.
datos = B.preparar_datos(largo, col_n="n", min_anios=5, exigir_2022=True)
print("Escuelas:", datos["n_escuelas"], "| observaciones:", len(datos["y"]))

15:15:49 | INFO    | simce_panel.bayes | 5 escuelas excluidas por no tener puntaje en 2022.


15:15:49 | INFO    | simce_panel.bayes | Datos del modelo: 2861 escuelas, 22606 observaciones.


Escuelas: 2861 | observaciones: 22606


## 2. Ajuste del modelo de medición

**Revisión del 1-oct (comentarios 19 y 24).** El modelo pasa a ser el modelo principal del análisis. El lineal jerárquico frecuentista se conserva como comparación en el cuaderno 10. (La versión anterior decía que su varianza comunal se estimaba en cero: era un error de convergencia del optimizador, corregido el 24-oct-2026; ver cuaderno 05.) Usa una varianza de medición heterogénea: un establecimiento con pocos evaluados aporta una observación más ruidosa y se contrae más hacia su grupo.

**Priors.** Los parámetros de grupo usan normales débilmente informativas en la escala estandarizada (DE 50 puntos = 1). La dispersión entre estudiantes, σ_w, tiene un prior log-normal centrado en 1, porque la desviación individual de la escala SIMCE es de 50 puntos; σ_e, la variación del establecimiento entre años, tiene un prior media-normal de escala 0,5.

**Criterios de aceptación:** cero divergencias, R-hat ≤ 1,01 y tamaño efectivo mínimo (ESS) de al menos 200 en los parámetros de grupo.

In [3]:
# 1.000 + 1.000 iteraciones por cadena: con 500 + 500 el R-hat queda en 1,012,
# apenas sobre el umbral. En un equipo convencional tarda unos 6-8 minutos.
mcmc = B.ajustar(datos, tendencia_pre=True, num_warmup=1000,
                 num_samples=1000, num_chains=2)
diag = mcmc.diagnosticos
print(diag)
s = mcmc.get_samples()
print("sigma_w (escala estandarizada):", round(float(s["s_w"].mean()), 3),
      "| sigma_e:", round(float(s["s_e"].mean()), 3))
print("Convergencia aceptable:",
      diag["divergencias"] == 0 and diag["r_hat_max"] <= 1.01)

15:15:51 | INFO    | jax._src.xla_bridge | Unable to initialize backend 'tpu': UNIMPLEMENTED: LoadPjrtPlugin is not implemented on windows yet.


15:19:33 | INFO    | simce_panel.bayes | Ajuste: {'segundos': 221.7, 'divergencias': 0, 'r_hat_max': 1.002, 'n_eff_min': 770}


{'segundos': 221.7, 'divergencias': 0, 'r_hat_max': 1.002, 'n_eff_min': 770}
sigma_w (escala estandarizada): 1.162 | sigma_e: 0.168
Convergencia aceptable: True


In [4]:
# Convergencia de los parámetros de cada escuela (auditoría del 24-oct-2026).
# C2 exige R-hat <= 1,01; aquí se verifica también para las 2.861 caídas y pendientes.
diag_escuela = B.diagnosticos_escuela(mcmc)
print(diag_escuela)
print("C2 también por escuela:",
      all(v["r_hat_max"] <= 1.01 for v in diag_escuela.values()))

{'caida': {'r_hat_max': 1.004, 'n_eff_min': 1674}, 'recuperacion': {'r_hat_max': 1.003, 'n_eff_min': 2135}}
C2 también por escuela: True


## 3. Efectos por grupo socioeconómico (H3b)

Intervalos de credibilidad al 94 %. La columna `recuperacion_neta` descuenta la tendencia previa: es lo que el establecimiento ganó por sobre lo que ya venía mejorando.

In [5]:
grupos = B.resumen_grupos(mcmc, datos)
grupos["recuperacion_neta"] = (grupos["recuperacion"] - grupos["tendencia_pre"]).round(1)
grupos

,grupo,caida,caida_ic_inf,caida_ic_sup,recuperacion,recuperacion_ic_inf,recuperacion_ic_sup,tendencia_pre,tendencia_pre_ic_inf,tendencia_pre_ic_sup,recuperacion_neta
0,Bajo,-0.800000,-2.500000,0.8,3.1,2.5,3.6,1.6,1.2,2.0,1.5
1,Medio bajo,-7.900000,-9.200000,-6.6,3.4,2.9,3.8,0.8,0.4,1.1,2.6
2,Medio,-17.799999,-19.299999,-16.4,3.7,3.2,4.1,-0.5,-0.9,-0.1,4.2
3,Medio alto,-21.200001,-23.400000,-19.1,3.8,3.1,4.4,-0.9,-1.5,-0.3,4.7
4,Alto,-10.600000,-12.400000,-8.7,4.5,3.9,5.1,-0.0,-0.6,0.5,4.5


### Lectura

**H3 se contrasta en su formulación original** (revisión del 1-oct, comentario 15): la recuperación es más lenta a menor GSE. La evidencia relevante es la pendiente, controlando el nivel previo, no la caída.

**H3b (exploratoria, formulada post hoc).** La caída de 2022 presenta un gradiente no monótono por GSE, con mayor magnitud en los grupos medio y medio-alto. Coincide con el CIAE (2023), que observa el mismo patrón con datos censales de 2017–2018 y 2022.

El efecto piso queda descartado: el grupo Bajo promedia unos 220 puntos en el PRE. Las alternativas que siguen abiertas son una menor tasa de participación en 2022 en escuelas vulnerables y la exclusión diferencial por supresión (notebook 07).

In [6]:
p = grupos.set_index("grupo")
for g in ["Bajo", "Alto"]:
    contrafactual = p.loc[g, "caida"] - 5.5 * p.loc[g, "tendencia_pre"]
    print(f"{g:5s} caída vs. PRE: {p.loc[g,'caida']:6.1f} | "
          f"vs. su tendencia: {contrafactual:6.1f}")

Bajo  caída vs. PRE:   -0.8 | vs. su tendencia:   -9.6
Alto  caída vs. PRE:  -10.6 | vs. su tendencia:  -10.6


## 4. Escuela por escuela: por qué no aparecen conglomerados

El modelo entrega, para cada establecimiento, un intervalo de credibilidad de su recuperación. Esto explica el resultado nulo del análisis espacial (notebook 05).

In [7]:
escuelas = B.resumen_escuelas(mcmc, datos)
ancho_ic = (escuelas["recuperacion_ic_sup"] - escuelas["recuperacion_ic_inf"]).mean()
print(f"Escuelas con recuperación creíblemente distinta de cero: "
      f"{int(escuelas.recuperacion_creible.sum())} de {len(escuelas)} "
      f"({100*escuelas.recuperacion_creible.mean():.1f} %)")
print(f"Ancho medio del intervalo de la recuperación: {ancho_ic:.1f} puntos/año")

Escuelas con recuperación creíblemente distinta de cero: 511 de 2861 (17.9 %)
Ancho medio del intervalo de la recuperación: 11.1 puntos/año


Con un intervalo medio de unos 11 puntos por año para un efecto típico de 3 a 4, la pendiente individual de la mayoría de las escuelas **no es distinguible del estancamiento**. LISA contrasta escuela por escuela, de modo que no puede encontrar conglomerados: el ruido individual domina.

**Implicancia para la tesis.** La estructura territorial no debe buscarse contrastando escuelas aisladas, sino **modelándose en el nivel de grupo**: agregar efectos aleatorios por comuna o región sobre $d_i$ y $b_i$ (y, en Capstone II, un efecto espacial ICAR). Esa es la vía para reformular H1 y H2.

In [8]:
escuelas.to_csv(cfg.ruta_salida("bayes_escuelas_mate.csv"), sep=";",
                index=False, encoding="utf-8-sig")
grupos.to_csv(cfg.ruta_salida("bayes_grupos_mate.csv"), sep=";",
              index=False, encoding="utf-8-sig")
print("Guardado en", cfg.salida_dir)

Guardado en Dataset\Unificados


## 5. Chequeo predictivo posterior (comentario 19)

Se simulan datos desde el posterior y se comparan con los observados por año. La cobertura del intervalo predictivo al 94 % debería ser cercana a 0,94; valores mayores indican un modelo conservador.

In [9]:
ppc = B.chequeo_predictivo(mcmc, datos, tendencia_pre=True)
ppc

15:19:37 | INFO    | simce_panel.bayes | Cobertura global del intervalo predictivo al 94 %: 0.980


,anio,media_obs,media_rep,de_obs,de_rep,cobertura
0,2015,262.2,263.9,46.8,45.7,0.967
1,2016,264.9,263.9,46.7,45.1,0.980
2,2017,265.2,264.1,45.1,44.5,0.986
3,2018,263.5,264.3,43.2,43.8,0.969
4,2022,252.9,252.8,36.8,39.3,0.990
5,2023,257.7,256.6,39.1,39.5,0.979
6,2024,259.6,260.1,40.3,40.2,0.977
7,2025,263.8,263.7,40.5,41.2,0.991


## 6. Correlación caída–recuperación descontado el ruido (comentario 11)

`dentro_escuela` mide cuánta información comparten caída y pendiente de un mismo establecimiento (el puntaje de 2022). `entre_escuelas` es la relación sustantiva, ya separada de esa dependencia.

In [10]:
B.correlacion_caida_pendiente(mcmc)

{'entre_escuelas': 0.278, 'dentro_escuela_media': -0.422}

**Lectura.** Si la correlación cruda era negativa y la correlación entre escuelas resulta positiva, la aparente recuperación compensatoria ("los que más cayeron se recuperan más") era un artefacto de construcción: descontado el ruido, los que más cayeron se recuperan más lento. Las medias a posteriori están contraídas, por lo que la magnitud debe leerse con cautela; el signo es lo informativo.

## 7. Recuperación relativa con umbral de credibilidad (comentario 21)

Se calcula solo para establecimientos con P(caída < 0) ≥ 0,9. Lo recuperado se define como pendiente × 3.

In [11]:
rho = B.recuperacion_relativa_bayes(mcmc, datos, prob_minima=0.9, horizonte=3)
print(rho["clase"].value_counts().to_string())
print("Mediana de la recuperación relativa (caída creíble):",
      round(float(rho["rho_mediana"].median()), 3))
rho.to_csv(cfg.ruta_salida("bayes_recuperacion_relativa_mate.csv"), sep=";",
           index=False, encoding="utf-8-sig")

15:19:38 | INFO    | simce_panel.bayes | Caída creíble en 1324 de 2861 establecimientos (P >= 0.90).


clase
sin caída detectable    1537
caída creíble           1324
Mediana de la recuperación relativa (caída creíble): 0.396


## 8. Segmento rural (comentario 13)

Los establecimientos rurales son unos 150 en la muestra. El agrupamiento parcial estabiliza sus estimaciones, y sus intervalos deben reportarse siempre junto a la estimación puntual.

In [12]:
rur = ancho[["rbd", "cod_rural_rbd_cat"]].drop_duplicates("rbd")
esc_r = escuelas.merge(rur, on="rbd", how="left")
esc_r.groupby("cod_rural_rbd_cat")[["recuperacion_bayes", "recuperacion_ic_inf",
                                    "recuperacion_ic_sup"]].median().round(2)

,recuperacion_bayes,recuperacion_ic_inf,recuperacion_ic_sup
cod_rural_rbd_cat,,,
Rural,3.50,-2.25,9.35
Urbano,3.47,-2.07,9.02


## 9. Aproximación descartada al contraste de H3 (comentario 15)

H3: la recuperación es más lenta a menor GSE. Se contrasta con la pendiente estimada, controlando el puntaje previo, con errores estándar robustos (HC3).

**Cautela.** Las medias a posteriori ya están contraídas hacia la media de su GSE, lo que puede amplificar las diferencias entre grupos. La prueba definitiva incorpora el puntaje previo dentro de la ecuación de la pendiente del modelo; esta regresión es una aproximación.

**Esta sección no es la prueba de H3.** Se conserva como registro de un error metodológico. La prueba válida está en el notebook 09 (sección 2) y su robustez en el notebook 11.

In [13]:
import statsmodels.formula.api as smf
res = sp.io_datos.leer_csv(cfg.ruta_salida("resultados_mate_2m.csv"))
h3 = (escuelas.merge(ancho[["rbd", "cod_grupo_cat"]], on="rbd")
              .merge(res[["rbd", "puntaje_pre"]], on="rbd"))
h3["pre_c"] = (h3["puntaje_pre"] - h3["puntaje_pre"].mean()) / 50
fit = smf.ols('recuperacion_bayes ~ C(cod_grupo_cat, Treatment("Bajo")) + pre_c',
              data=h3).fit(cov_type="HC3")
fit.summary2().tables[1][["Coef.", "[0.025", "0.975]", "P>|z|"]].round(3)

,Coef.,[0.025,0.975],P>|z|
Intercept,2.619,2.418,2.819,0.0
"C(cod_grupo_cat, Treatment(""Bajo""))[T.Alto]",2.498,2.111,2.884,0.0
"C(cod_grupo_cat, Treatment(""Bajo""))[T.Medio]",1.166,0.893,1.440,0.0
"C(cod_grupo_cat, Treatment(""Bajo""))[T.Medio alto]",1.457,1.098,1.816,0.0
"C(cod_grupo_cat, Treatment(""Bajo""))[T.Medio bajo]",0.525,0.309,0.741,0.0
pre_c,-0.520,-0.665,-0.374,0.0


**Lectura (corregida).** Estos coeficientes **no** respaldan H3. Las medias a posteriori ya están contraídas hacia la media de su GSE, de modo que regresionarlas sobre el GSE recupera la estructura del propio modelo, con intervalos artificialmente estrechos: el gradiente es un artefacto.

La prueba válida incorpora el puntaje previo dentro de la ecuación de la pendiente (notebook 09, sección 2). Con ella, en Matemática no hay gradiente (GSE Alto − Bajo: +0,09 puntos/año, IC 94 % [−1,07; 1,23]); en Lectura sí (+1,53 [0,64; 2,40]), pero no resiste los controles del notebook 11. **H3 no se sostiene.** H3b (gradiente no monótono de la caída) se reporta aparte, rotulada como exploratoria.

## 10. Sensibilidad: GSE previo a la pandemia (auditoría del 24-oct-2026)

El GSE de la Agencia se recalcula cada año, y el panel usa el del año más reciente. Para un análisis de la caída de 2022 eso es problemático: el GSE posterior puede reflejar cambios ocurridos después del tratamiento. Aquí se usa el **GSE modal de 2015–2018** de cada escuela (las que no tienen GSE previo conservan el más reciente) y se repite el ajuste.

In [14]:
largo_rbd = sp.io_datos.leer_csv(cfg.ruta_salida(sp.config.NOMBRES_SALIDA["rbd"]))
if "cod_grupo_cat" not in largo_rbd.columns:
    largo_rbd, _ = sp.estandarizacion.estandarizar_canonico(largo_rbd, variables=["cod_grupo"])
gse_pre = B.gse_modal_previo(largo_rbd)

en_modelo = pd.Index(np.asarray(datos["rbd"]))
comp = pd.DataFrame({"previo": gse_pre.reindex(en_modelo),
                     "reciente": ancho.set_index("rbd")["cod_grupo_cat"].reindex(en_modelo)})
distinto = comp.previo.notna() & (comp.previo != comp.reciente)
print(f"Escuelas con GSE previo distinto del más reciente: {int(distinto.sum())} de {len(comp)} "
      f"({100 * distinto.mean():.1f} %)")
pd.crosstab(comp.previo, comp.reciente)

Escuelas con GSE previo distinto del más reciente: 719 de 2861 (25.1 %)


reciente,Alto,Bajo,Medio,Medio alto,Medio bajo
previo,,,,,
Alto,365,0,1,5,0
Bajo,0,482,6,2,181
Medio,5,0,487,81,85
Medio alto,64,0,30,267,0
Medio bajo,0,97,161,1,541


In [15]:
largo_pre = largo.assign(cod_grupo_cat=largo["rbd"].map(gse_pre).fillna(largo["cod_grupo_cat"]))
datos_pre = B.preparar_datos(largo_pre, col_n="n", min_anios=5, exigir_2022=True)
mcmc_pre = B.ajustar(datos_pre, tendencia_pre=True, num_warmup=1000, num_samples=1000, num_chains=2)
print(mcmc_pre.diagnosticos)
grupos_pre = B.resumen_grupos(mcmc_pre, datos_pre)
p2 = grupos_pre.set_index("grupo")
for g in ["Bajo", "Alto"]:
    print(f"{g:5s} caída vs. PRE: {p2.loc[g, 'caida']:6.1f} | "
          f"vs. su tendencia: {p2.loc[g, 'caida'] - 5.5 * p2.loc[g, 'tendencia_pre']:6.1f}")
grupos_pre

15:19:41 | INFO    | simce_panel.bayes | 5 escuelas excluidas por no tener puntaje en 2022.


15:19:42 | INFO    | simce_panel.bayes | Datos del modelo: 2861 escuelas, 22606 observaciones.


15:22:48 | INFO    | simce_panel.bayes | Ajuste: {'segundos': 186.4, 'divergencias': 0, 'r_hat_max': 1.003, 'n_eff_min': 775}


{'segundos': 186.4, 'divergencias': 0, 'r_hat_max': 1.003, 'n_eff_min': 775}
Bajo  caída vs. PRE:    0.7 | vs. su tendencia:   -9.8
Alto  caída vs. PRE:   -9.8 | vs. su tendencia:  -10.4


,grupo,caida,caida_ic_inf,caida_ic_sup,recuperacion,recuperacion_ic_inf,recuperacion_ic_sup,tendencia_pre,tendencia_pre_ic_inf,tendencia_pre_ic_sup
0,Bajo,0.700000,-0.700000,2.1,3.4,3.0,3.9,1.9,1.5,2.3
1,Medio bajo,-8.100000,-9.500000,-6.8,3.4,3.0,3.8,0.7,0.4,1.1
2,Medio,-20.299999,-21.700001,-18.9,3.4,3.0,3.9,-0.7,-1.1,-0.3
3,Medio alto,-22.500000,-24.700001,-20.5,3.7,3.1,4.3,-1.3,-1.9,-0.8
4,Alto,-9.800000,-11.700000,-7.7,4.5,3.8,5.1,0.1,-0.5,0.6


**Lectura.** Si el GSE Bajo sigue mostrando una tendencia previa positiva y una caída casi nula frente a su promedio previo, el hallazgo principal no depende de qué GSE se use. Salida esperada: Bajo +0,7 [−0,8; 2,1] frente a su promedio previo y −9,8 frente a su tendencia (antes −0,8 y −9,6); Alto −10,4. Las escuelas que pasaron de Bajo a Medio bajo después de 2018 (unas 180) venían subiendo, y con el GSE reciente "salían" del grupo Bajo; con el GSE previo vuelven a él y el resultado se mantiene.